In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../..')

from Model.Combined.DataPrep.Data_Prep import Combined_Data_Prep
from Model.Combined.DataPrep.Player_Dataset import Create_Test_Train_Datasets
from Model.Pro.DataPrep import Prep_Map, Output_Map
from Model.College.DataPrep import Prep_Map as C_Prep_Map, Output_Map as C_Output_Map

In [ ]:
is_hitter = True

In [ ]:
from Model.Constants import DATA_PREP_BINARY_FILE

# data_prep = Combined_Data_Prep(
#     Prep_Map.base_prep_map, 
#     Output_Map.base_output_map,
#     C_Prep_Map.college_base_prep_map,
#     C_Output_Map.college_output_map,
#     save_name="../" + DATA_PREP_BINARY_FILE)

data_prep = Combined_Data_Prep.Load_From_File("../" + DATA_PREP_BINARY_FILE)

In [ ]:
io_list = data_prep.Generate_IO_Hitters(is_training=True) if is_hitter else \
    data_prep.Generate_IO_Pitchers(is_training=True)

In [ ]:
train_dataset, test_dataset = Create_Test_Train_Datasets(
    player_list=io_list, 
    is_hitter=is_hitter)

In [ ]:
from Model.Pro.Model.Player_Model import RNN_Model as Pro_Model
from Model.College.Model.College_Model import RNN_Model as College_Model

import torch
from Model.Constants import device

pro_model = Pro_Model(
    input_size=data_prep.GetProIOSize(is_hitter),
    mutators=torch.empty(0),
    data_prep=data_prep.pro_data_prep,
    is_hitter=is_hitter,
).to(device)
col_model = College_Model(
    input_size=data_prep.GetColIoSize(is_hitter),
    data_prep=data_prep.college_data_prep,
    is_hitter=is_hitter,
    output_hidden_size=pro_model.GetHiddenSize(),
    output_num_layers=pro_model.GetNumLayers(),
).to(device)

In [ ]:
from Model.Combined.Model.Model_Train import TrainAndGraph

train_results = TrainAndGraph(
    pro_network=pro_model,
    col_network=col_model,
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    pro_model_name="../Models/test_pro",
    col_model_name="../Models/test_col",
    is_hitter=is_hitter,
)